Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

Read Bronze table

In [0]:
df = spark.table("workspace.bronze.prd_info")

In [0]:
df.display()

Silver Transformations

Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))
  

In [0]:
df.display()

Product Key Parsing

Cost Cleanup

Product Line Normalization

In [0]:
df=(
    df.
    withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M","Mountain")
        .when(F.upper(col("prd_line")) =="R","Road")
        .when(F.upper(col("prd_line")) =="T","Touring")
        .when(F.upper(col("prd_line")) =="S","Other Sales")
        .otherwise("n/a")
        
    )
)

Date Casting

In [0]:
df=df.withColumn("prd_start_dt",col("prd_start_dt").cast(DateType()))

Renaming Columns

In [0]:
RENAME_MAP={
    "prd_id":"product_id",
    "prd_key":"product_key",
    "prd_nm":"product_name",
    "prd_cost":"product_cost",
    "prd_line":"product_line",
    "prd_start_dt":"product_start_dt",
    "prd_end_dt":"product_end_dt"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

Sanity checks of dataframe

In [0]:
df.limit(10).display()

Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

Sanity checks of Silver table